In [3]:
import torch
from datasets import load_dataset
from torchvision.transforms import v2
import dotenv
import os

In [7]:
dotenv.load_dotenv()
#print(os.getenv("HF_TOKEN")) #HF_TOKEN prescence in envvars is automatically detected by huggingface functions

True

In [ ]:
hf_dataset_training = load_dataset("cifar10", split="train[:5%]")
hf_dataset_val_test = load_dataset("cifar10", split="test[:2%]")

train_transform = v2.Compose([
    v2.ToImage(), # converts into Image
    v2.RandomCrop(32, padding=4), # randomly crops a portion of the image 
    v2.RandomHorizontalFlip(p=0.5), #mirror 50% of the images
    v2.ToDtype(torch.float32, scale=True),# converts to tensor with datatype float32 
    v2.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2023, 0.1994, 0.2010]) #z-score normalization for the 3-channels
])

class HuggingFaceCIFAR(torch.utils.data.Dataset):
    def __init__(self, hf_dataset, transform):
        self.dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx] #{label, image}
        image = self.transform(item['img']) #apply transform to images independently
        label = item['label']
        return image, label

dataset_trani = HuggingFaceCIFAR(hf_dataset_training, train_transform)
train_loader = torch.utils.data.DataLoader(small_dataset, batch_size=64, shuffle=True)
